# ★ Додаткове завдання: власний набір даних, `merge`/`concat`, групування, зведена таблиця
**Виконав:** Мельниківський Андрій, група КН-41

**Сценарій:** інтернет-магазин електроніки. Дві таблиці: `products` (довідник товарів) та `orders` (замовлення за два місяці). Результати зберігаються в папку `results/` — її потрібно завантажити в репозиторій GitHub.

In [1]:
import os
import pandas as pd
import numpy as np
os.makedirs('results', exist_ok=True)

### 1. Створення двох таблиць

In [2]:
products = pd.DataFrame({
    'product_id': [1, 2, 3, 4, 5, 6],
    'product': ['Смартфон', 'Ноутбук', 'Навушники', 'Планшет', 'Миша', 'Клавіатура'],
    'category': ['Гаджети', 'Комп\'ютери', 'Аксесуари', 'Гаджети', 'Аксесуари', 'Аксесуари'],
    'price': [15000, 32000, 2500, 12000, 600, 1200]
})

rng = np.random.default_rng(5)
def make_orders(start_id, month, n):
    return pd.DataFrame({
        'order_id': range(start_id, start_id + n),
        'product_id': rng.integers(1, 7, n),
        'city': rng.choice(['Київ', 'Львів', 'Тернопіль', 'Одеса'], n),
        'quantity': rng.integers(1, 4, n),
        'month': month
    })

orders_mar = make_orders(1, 'березень', 40)
orders_apr = make_orders(41, 'квітень', 40)
products

,product_id,product,category,price
0,1,Смартфон,Гаджети,15000
1,2,Ноутбук,Комп'ютери,32000
2,3,Навушники,Аксесуари,2500
3,4,Планшет,Гаджети,12000
4,5,Миша,Аксесуари,600
5,6,Клавіатура,Аксесуари,1200


### 2. `concat`: об'єднуємо замовлення двох місяців

In [3]:
orders = pd.concat([orders_mar, orders_apr], ignore_index=True)
print('Замовлень:', len(orders_mar), '+', len(orders_apr), '=', len(orders))
orders.head()

Замовлень: 40 + 40 = 80


,order_id,product_id,city,quantity,month
0,1,5,Одеса,2,березень
1,2,5,Тернопіль,2,березень
2,3,1,Одеса,1,березень
3,4,5,Львів,3,березень
4,5,3,Львів,3,березень


### 3. `merge`: приєднуємо довідник товарів

In [4]:
data = orders.merge(products, on='product_id', how='left')
data['revenue'] = data['price'] * data['quantity']
print('Рядків без товару:', data['product'].isnull().sum())
data.head()

Рядків без товару: 0


,order_id,product_id,city,quantity,month,product,category,price,revenue
0,1,5,Одеса,2,березень,Миша,Аксесуари,600,1200
1,2,5,Тернопіль,2,березень,Миша,Аксесуари,600,1200
2,3,1,Одеса,1,березень,Смартфон,Гаджети,15000,15000
3,4,5,Львів,3,березень,Миша,Аксесуари,600,1800
4,5,3,Львів,3,березень,Навушники,Аксесуари,2500,7500


### 4. Групування та агрегатні показники (сума, середнє, кількість)

In [5]:
stats = data.groupby('category')['revenue'].agg(total_revenue='sum', avg_order='mean', orders_count='count').round(2)
stats

,total_revenue,avg_order,orders_count
category,,,
Аксесуари,107800,2836.84,38
Гаджети,759000,25300.00,30
Комп'ютери,800000,66666.67,12


In [6]:
by_city = data.groupby('city').agg(total_revenue=('revenue', 'sum'), avg_quantity=('quantity', 'mean'), orders_count=('order_id', 'count')).round(2)
by_city

,total_revenue,avg_quantity,orders_count
city,,,
Київ,488400,2.24,17
Львів,429800,1.72,25
Одеса,395000,2.05,20
Тернопіль,353600,2.17,18


### 5. Зведена таблиця: категорія × місяць (сума доходу)

In [7]:
pivot = pd.pivot_table(data, values='revenue', index='category', columns='month', aggfunc='sum', fill_value=0, margins=True, margins_name='Разом')
pivot

month,березень,квітень,Разом
category,,,
Аксесуари,52800,55000,107800
Гаджети,363000,396000,759000
Комп'ютери,544000,256000,800000
Разом,959800,707000,1666800


### 6. Збереження результатів у CSV

In [8]:
data.to_csv('results/orders_merged.csv', index=False)
stats.to_csv('results/stats_by_category.csv')
by_city.to_csv('results/stats_by_city.csv')
pivot.to_csv('results/pivot_category_month.csv')
print(sorted(os.listdir('results')))

['orders_merged.csv', 'pivot_category_month.csv', 'stats_by_category.csv', 'stats_by_city.csv']


### Висновок
Поєднано дві таблиці (`concat` + `merge`), обчислено три види агрегатів (сума, середнє, кількість) за категоріями та містами й побудовано зведену таблицю «категорія × місяць». Результати збережено у `results/`.